# Module 7: Expand the Candidate Sample and Screen Financial Coverage

目标：从 3 家试点扩展到 Module 1 的候选公司清单，下载其 SEC Company Facts，并检查实际年度数据的覆盖情况。
本模块接在现有 Notebook 后运行。教学解释用中文；代码、注释、列名、状态与打印信息用英文。
继续使用普通循环和 pandas；不引入回归模型或自定义财务处理函数。

这一步回答三个问题：哪些公司成功取得数据？哪些年度能取得研究所需指标？哪些公司值得进入会计口径审核？
42 家是候选池，不是 42 家已通过审核的研究公司。暂时不能保证最终会有 30–50 家合格公司。
公司名称与股票代码来自既有身份快照，本模块不验证其当前上市状态。历史样本仍可能受到存续公司选择偏差影响。

输入：retail_candidate_universe.csv、pilot_annual_panel_metadata.json、已有 Company Facts 缓存。
输出：下载日志、初步年度财务表、字段来源、公司覆盖摘要、待审核工作表与 metadata。
已有 3 家的缓存会复用；其余公司各需要一次请求。若此前正好有 3 份有效缓存，首次最多新增 39 次请求。
采用每次网络请求后暂停 1 秒的顺序下载；遇到 HTTP 403 或 429 时停止本轮后续网络请求，仍可读取已有缓存。

本模块不修改 pilot_ 开头的数据，也不批量套用 AEO/ANF 的试点推导公式。
因此初步表中这两家公司的部分缺失可能再次出现：它反映候选标准标签的直接覆盖，需要下一步按公司审核和推导。
SEC Company Facts 不包含公司自定义标签；缺失也可能来自标签选择、同文件匹配或历史覆盖，不能解释成公司没有披露。
下一模块将在这份工作表基础上审核业务与会计口径，再确定样本及研究区间。

来源：[SEC XBRL API](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、[SEC Developer Resources](https://www.sec.gov/about/developer-resources)。


本版修订：检查实际 42 份缓存后，补充 TJX/RH/TLYS 的收入候选及五家公司的存货候选。
仅扩展候选提取规则，所有金额继续标 selected_pending_review；没有完成全部历史年度的会计口径认证。
可直接运行本 Guide 的 8 格，全部缓存已存在时不会重新请求；也可替换原 Module 7 的 Cell 4 和 Cell 8，随后从 Cell 4 顺序重跑到 Cell 8。


本次用你的 42 份真实缓存验证后的参考结果：656 个年度记录；42 家达到至少 5 个核心金额完整年度的审核优先级；35 家至少有 5 个六指标非空年度。
这三个数量都不是最终研究资格认证。存货缺失从原版的 132 行降至 63 行，剩余缺失继续保留。
毛利缺失数量可能增加，因为收入标签补全后新增了原来没有进入表的年度，而这些新增年度的毛利仍可能需要人工提取或核实推导。
以上参考值取决于本次缓存版本；以后刷新数据、扩展公司或修改规则后，数量变化并不一定意味着代码出错。


## Cell 1: Set Up the Tools and Directories

导入前面用过的工具。requests 负责获取网页接口返回的 JSON，time.sleep 用来控制请求间隔。
如果提示 requests 或 pandas 尚未安装，单独运行 `%pip install requests pandas`，再执行本格。

路径必须与之前的模块一致。这里同时需要读取原始缓存和已处理的公司清单。
本模块使用 Module 2 已设置的 SEC_HEADERS，不需要重新填写邮箱；它应包含你的姓名及联系邮箱。
如果重启过 Kernel，请先重新执行 Module 2 中定义 SEC_HEADERS 的配置格，无须重新执行试点下载。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import requests
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read the Candidate Pool

读取 Module 1 的清单，只处理 identity_status 为 matched、且有 CIK 的行。
这个条件确认“能够对应到 SEC 身份”，并不等于 eligibility_status 已合格。
保留 subgroup，后续可以按业务类型审核，而不是把服装、家居、电子产品等公司直接混在一起比较。

cik10 保持为字符串，前导零不会丢失。drop_duplicates 防止同一股票代码重复下载。
MIN_CORE_PERIODS=5 是事先设定的审核优先级：至少有 5 个核心金额完整的年度，才先进入口径审核。
它不是最终纳入标准，也不保证这些年度连续、能够形成 4 个合格的下一年配对。
这次沿用较宽的历史覆盖做筛查；正式分析区间将在样本审核后明确。


In [ ]:
universe = pd.read_csv(
    PROCESSED_DIR / "retail_candidate_universe.csv", dtype={"cik10": "string"}
)
candidate_pool = universe.loc[
    universe["identity_status"].eq("matched") & universe["cik10"].notna()
].drop_duplicates("ticker").copy()
MIN_CORE_PERIODS = 5
print("Identity-matched candidates:", len(candidate_pool))
display(candidate_pool.groupby("subgroup", sort=False).size().rename("Candidate Count"))


## Cell 3: Download or Reuse Each Company Snapshot

这是本模块唯一发送网络请求的格。请先确认 SEC_HEADERS 已按 Cell 1 的说明设置。

循环处理每家公司：已有缓存就读取；没有缓存才请求。raise_for_status() 把 HTTP 失败转成异常，避免把错误页面当财务数据。
time.sleep(1) 放在 finally 中，保证请求失败时也会暂停；这与下方记录公司失败的 except 分工不同。
下载得到的 payload 通过 CIK 和 facts 结构核对后，才保存为与 Module 2 相同格式的 snapshot。
retrieved_at_utc 记录获取时间，复用缓存时保留原始时间，不把“读取旧缓存”伪装成“今天获取”。

try/except 的用途是让一家公司的网络或文件问题不终止整批处理。error 列留下原因。
403/429 通常需要处理访问或频率问题，因此 download_allowed=False 会暂停本轮后面的新请求。
not_attempted 表示尚未尝试；failed 表示本家公司出错。都不能当作财务数据缺失，更不能删除后假装样本已经完整。
已有缓存损坏时会记 failed，不自动覆盖；先检查日志，再单独处理该文件。成功的新缓存保存后可直接复用，重新运行不会再次下载它们。


In [ ]:
all_company_facts = {}
snapshot_times = {}
download_rows = []
download_allowed = True
for index, row in candidate_pool.iterrows():
    ticker = row["ticker"]
    cache_path = FACTS_DIR / f"CIK{row['cik10']}_companyfacts.json"
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{row['cik10']}.json"
    status, error, snapshot = "not_attempted", "", None
    try:
        if cache_path.exists():
            snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
            status = "cached"
        elif download_allowed:
            try:
                response = requests.get(url, headers=SEC_HEADERS, timeout=30)
            finally:
                time.sleep(1)
            if response.status_code in [403, 429]:
                download_allowed = False
            response.raise_for_status()
            snapshot = {
                "source_url": url,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                "payload": response.json(),
            }
            status = "downloaded"
        if snapshot is not None:
            payload = snapshot["payload"]
            if int(payload["cik"]) != int(row["cik10"]) or not isinstance(payload["facts"], dict):
                raise ValueError("Invalid Company Facts payload or CIK mismatch.")
            if status == "downloaded":
                cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
            snapshot_times[ticker] = snapshot["retrieved_at_utc"]
            all_company_facts[ticker] = payload
    except (requests.RequestException, ValueError, KeyError, OSError) as exc:
        status, error = "failed", str(exc)
    download_rows.append({"ticker": ticker, "download_status": status, "error": error})
    print(ticker, status)
download_log = pd.DataFrame(download_rows)
display(download_log["download_status"].value_counts())
display(download_log.loc[download_log["download_status"].isin(["failed", "not_attempted"])])


## Cell 4: Collect Candidate Financial Records

从已保存的 Module 3 metadata 读取 tag_candidates_in_priority_order，复用你已修正的标签清单，避免手工重抄长字典。
请确认显示的 cost_of_sales 候选包括之前补充的两个 ExcludingDepreciation 标签。
它们只是候选，不代表不同公司的成本口径相同；operating_cash_flow 的持续经营口径也需要审核。

三层循环把成功加载的公司、指标、标签逐条展开成 candidate_facts 长表，并保留文件编号、日期和金额。
只取 USD；如果公司用其他货币或其他会计准则，这一版本的筛查可能无法覆盖，后续应单独处理。
数据完全为空时停止，提醒先处理下载或标签问题。drop_duplicates 只去掉完全相同的记录。


此次需要增加 company_tag_additions：先按指标、再按股票代码保存额外候选标签。
extra_tags.get(metric, {}).get(ticker, []) 的含义是：找到本指标对应的公司字典，再找该公司的额外标签；没有配置就返回空列表。
base_tags + ... 创建本次循环使用的新列表，不修改原始试点标签字典。候选优先级仍保留原标签在前。

收入方面，TJX、RH、TLYS 新年度使用 RevenueFromContractWithCustomerIncludingAssessedTax。
仅给这三家增加候选；最终审核仍须核对报表收入及税收政策，不能把标签名称当作税额已计入报表的证明。
存货方面，按公司指定替代标签，避免把任意成品存货、存货变动额、估值准备或子项目当成所有公司的总存货。

已核对的最新年度存货金额（2026-01-31，USD）：GAP 2,207,000,000；DDS 1,201,098,000；RH 缓存候选 818,550,000；SPWH 312,858,000；ULTA 2,181,127,000。
RH 这里只按缓存标签定义列为候选，仍需对照原年报；其余四个金额已与年报资产负债表核对。
特别是 ULTA：InventoryFinishedGoods 的通用定义称 Gross，但该公司最新记录金额与报表 Merchandise inventories, net 相同。
因此本版仅为 ULTA 增加这条候选，不做任何减准备金运算；其历史年度仍须逐一审核。
GAP 的早期 InventoryFinishedGoods 本版继续不自动替代净存货，避免把历史毛额和净额混用。

核对来源：[GAP 2026 10-K](https://www.sec.gov/Archives/edgar/data/39911/000162828026018573/gap-20260131.htm)、[DDS 2026 10-K](https://www.sec.gov/Archives/edgar/data/28917/000002891726000006/dds-20260131x10k.htm)、[ULTA 2026 10-K](https://www.sec.gov/Archives/edgar/data/1403568/000110465926035243/ulta-20260131x10k.htm)、[SPWH 年报](https://www.sec.gov/Archives/edgar/data/1132105/000119312526154924/spwh-ars-2026.pdf)。


In [ ]:
pilot_metadata = json.loads(
    (PROCESSED_DIR / "pilot_annual_panel_metadata.json").read_text(encoding="utf-8")
)
tag_candidates = pilot_metadata["tag_candidates_in_priority_order"]
metrics = list(tag_candidates)
company_tag_additions = {
    "revenue": {
        "TJX": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
        "RH": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
        "TLYS": ["RevenueFromContractWithCustomerIncludingAssessedTax"],
    },
    "inventory": {
        "GAP": ["InventoryFinishedGoodsNetOfReserves"],
        "DDS": ["RetailRelatedInventoryMerchandise"],
        "RH": ["RetailRelatedInventoryMerchandise"],
        "SPWH": ["RetailRelatedInventory"],
        "ULTA": ["InventoryFinishedGoods"],
    },
}
print("Cost tag candidates:", tag_candidates["cost_of_sales"])
rows = []
for ticker, data in all_company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, base_tags in tag_candidates.items():
        possible_tags = base_tags + company_tag_additions.get(metric, {}).get(ticker, [])
        for priority, tag in enumerate(possible_tags):
            for fact in tags.get(tag, {}).get("units", {}).get("USD", []):
                rows.append({
                    "ticker": ticker, "metric": metric, "tag": tag, "priority": priority,
                    "start": fact.get("start"), "end": fact["end"], "value": fact["val"],
                    "form": fact["form"], "filed": fact["filed"], "accn": fact["accn"],
                })
candidate_facts = pd.DataFrame(rows).drop_duplicates()
if candidate_facts.empty:
    raise ValueError("No candidate USD facts found. Review the download log and tags.")
for column in ["start", "end", "filed"]:
    candidate_facts[column] = pd.to_datetime(candidate_facts[column])
display(candidate_facts.head())


## Cell 5: Identify Annual Revenue Anchors

沿用 Module 3：收入记录来自 10-K/10-K/A，实际期间为 350–380 天。
按公司、年度终点、提交日期排序，为每个年度保留缓存中最早可见的合格收入记录。
短过渡年度会被这条规则排除，需要后续查年报；不能因为没被选中就认定没有披露。

candidate_anchors 是其他指标的匹配基准。其他金额必须来自同一份文件、同一年度范围，避免随意拼接后来修订的数字。
filed 仍只是 SEC 提交日期；本格没有证明这是首次公开金额，也没有认证历史数据无前视偏差。


In [ ]:
candidate_facts["period_days"] = (
    candidate_facts["end"] - candidate_facts["start"]
).dt.days + 1
annual_revenue = candidate_facts.loc[
    candidate_facts["metric"].eq("revenue")
    & candidate_facts["form"].isin(["10-K", "10-K/A"])
    & candidate_facts["period_days"].between(350, 380)
]
candidate_anchors = annual_revenue.sort_values(["ticker", "end", "filed", "accn", "priority"])
candidate_anchors = candidate_anchors.drop_duplicates(["ticker", "end"], keep="first")
if candidate_anchors.empty:
    raise ValueError("No annual revenue anchors found. Review the candidate revenue tags.")
print("Companies with annual revenue anchors:", candidate_anchors["ticker"].nunique())
print("Candidate annual periods:", len(candidate_anchors))


## Cell 6: Build the Preliminary Annual Panel

与 Module 3 相同：每个年度一行，每个指标在同公司、同文件编号和同期间终点内寻找。
存货与资产是时点值，额外要求 start 为空，避免选中带期间范围的记录；期间值要求 start 与年度收入相同。
按标签顺序选取后，金额唯一才填入；冲突留空并标 ambiguous，没有记录标 missing。

来源单独保存在 candidate_sources，下一模块可以直接定位缺失或冲突字段。
这里不推导毛利或销售成本。特别是 URBN 已有收入减成本与报表毛利不完全相等的案例，不能向 42 家直接推广公式。
selected_pending_review 只表示成功取值，不表示业务或费用范围已经审核。金额保留美元原值。


In [ ]:
annual_rows, source_rows = [], []
for index, year in candidate_anchors.iterrows():
    annual_row = {
        "ticker": year["ticker"], "period_start": year["start"], "period_end": year["end"],
        "period_days": year["period_days"], "filed": year["filed"], "accn": year["accn"],
    }
    for metric in metrics:
        matches = candidate_facts.loc[
            candidate_facts["ticker"].eq(year["ticker"]) & candidate_facts["metric"].eq(metric)
            & candidate_facts["end"].eq(year["end"]) & candidate_facts["accn"].eq(year["accn"])
        ]
        if metric in ["inventory", "total_assets"]:
            matches = matches.loc[matches["start"].isna()]
        else:
            matches = matches.loc[matches["start"].eq(year["start"])]
        value, tag, status = None, None, "missing"
        if not matches.empty:
            tag = matches.sort_values("priority").iloc[0]["tag"]
            chosen = matches.loc[matches["tag"].eq(tag)]
            if chosen["value"].nunique() == 1:
                value, status = chosen.iloc[0]["value"], "selected_pending_review"
            else:
                status = "ambiguous"
        annual_row[metric] = value
        source_rows.append({
            "ticker": year["ticker"], "period_end": year["end"], "metric": metric,
            "tag": tag, "value": value, "status": status, "accn": year["accn"],
        })
    annual_rows.append(annual_row)
candidate_panel = pd.DataFrame(annual_rows).sort_values(["ticker", "period_end"])
candidate_sources = pd.DataFrame(source_rows)
display(candidate_panel.head())


## Cell 7: Summarize Coverage and Create the Review Worksheet

core_complete 检查收入、存货、资产、经营现金流四个核心金额都存在；six_metric_complete 再要求销售成本与毛利存在。
这些标记只检查是否为空，不判断金额合理性、年度连续性、费用口径、历史重列或是否可用于下一年配对。
因此至少 5 个 core_complete 年度的公司仅被标 ready_for_accounting_review，仍须逐家公司查报告。
其余有年度数据的公司标 needs_metric_review；没有收入年度锚点与下载失败单独记录。
毛利或成本缺失并不自动排除公司，它可能通过报告提取或经过核实的公司专用推导规则解决。

groupby(...).agg(...) 按公司统计年度数量和每个指标非空的年度数量；sum 对 True/False 求和相当于计数。
最后用 left merge 把这些统计放回整个候选池。下载失败的公司仍在表里，未知覆盖数量保持为空，不冒充 0。
会计审核和最终纳入决定统一为 pending；没有公司在本模块自动获得最终资格。
显示公司层面的缺失概览，帮助区分“仅缺毛利”“缺历史存货”“没有下载成功”等不同问题。


In [ ]:
coverage_periods = candidate_panel[["ticker", "period_end"]].copy()
for metric in metrics:
    coverage_periods[metric] = candidate_panel[metric].notna()
coverage_periods["core_complete"] = candidate_panel[
    ["revenue", "inventory", "total_assets", "operating_cash_flow"]
].notna().all(axis=1)
coverage_periods["six_metric_complete"] = candidate_panel[metrics].notna().all(axis=1)
company_coverage = coverage_periods.groupby("ticker").agg(
    annual_periods=("period_end", "size"), first_period_end=("period_end", "min"),
    last_period_end=("period_end", "max"), core_complete_periods=("core_complete", "sum"),
    six_metric_complete_periods=("six_metric_complete", "sum"),
)
for metric in metrics:
    company_coverage[f"{metric}_periods"] = coverage_periods.groupby("ticker")[metric].sum()
coverage = candidate_pool[["ticker", "subgroup", "cik10", "sec_company_name"]].merge(
    download_log, on="ticker", how="left"
).merge(company_coverage, on="ticker", how="left")
coverage["coverage_status"] = "download_not_ready"
loaded = coverage["download_status"].isin(["cached", "downloaded"])
coverage.loc[loaded, "coverage_status"] = "no_annual_revenue_anchor"
coverage.loc[coverage["annual_periods"].notna(), "coverage_status"] = "needs_metric_review"
coverage.loc[coverage["core_complete_periods"].ge(MIN_CORE_PERIODS), "coverage_status"] = "ready_for_accounting_review"
review_sheet = coverage.copy()
review_sheet["accounting_review_status"] = "pending"
review_sheet["include_decision"] = "pending"
review_sheet["business_scope_notes"] = ""
review_sheet["cost_and_gross_profit_notes"] = ""
print("Final company eligibility remains pending.")
display(coverage["coverage_status"].value_counts())
display(coverage[["ticker", "annual_periods", "core_complete_periods", "six_metric_complete_periods", "coverage_status"]])


## Cell 8: Save the Screening Outputs

保存 candidate_ 开头的工作表，不覆盖已经检查过的 pilot_ 文件。
1. candidate_download_log.csv：缓存读取、下载、失败和未尝试状态。
2. candidate_annual_panel_preliminary.csv：按直接候选标签取值的初步年度金额。
3. candidate_field_sources.csv：每个金额的标签、文件编号和选择状态。
4. candidate_financial_coverage.csv：整个候选池的公司覆盖情况。
5. candidate_scope_review.csv：后续人工审核的工作表。重跑时更新覆盖统计，并按 ticker 保留已有的人工审核字段。
6. candidate_module7_metadata.json：规则、缓存获取时间及研究状态。

自动生成的 CSV 和 metadata 在重跑时更新。已有审核表以 ticker 合并到本轮候选池，当前覆盖统计更新、人工审核状态及备注保留。备注列读取为 string，避免全空列被 pandas 识别为浮点数。新加入的公司状态为 pending。修改提取规则后应重新检查相关公司的审核结论。
已有缓存的日期可能不同，metadata 如实保留。这里不自动刷新缓存，以免混淆试点来源和获取时间。

运行完成后，主要查看 Cell 3 的失败列表及 Cell 7 的覆盖摘要；不要只看下载是否全部成功。
下一步需要解释每家公司缺失的原因、核对业务与成本口径、标记不连续/重列年度，再生成扩大样本的特征和配对。
报告应分别陈述候选数量、成功下载数量、通过人工审核的公司数量和最终年度配对数量。


In [ ]:
output_tables = {
    "candidate_download_log.csv": download_log,
    "candidate_annual_panel_preliminary.csv": candidate_panel,
    "candidate_field_sources.csv": candidate_sources,
    "candidate_financial_coverage.csv": coverage,
}
for filename, table in output_tables.items():
    table.to_csv(PROCESSED_DIR / filename, index=False)
review_path = PROCESSED_DIR / "candidate_scope_review.csv"
review_columns = [
    "accounting_review_status", "include_decision",
    "business_scope_notes", "cost_and_gross_profit_notes",
]
if review_path.exists():
    previous_review = pd.read_csv(
        review_path, dtype={column: "string" for column in review_columns}
    )
    review_sheet = coverage.merge(
        previous_review[["ticker"] + review_columns], on="ticker", how="left", validate="one_to_one"
    )
    for column in review_columns:
        default = "pending" if column in ["accounting_review_status", "include_decision"] else ""
        review_sheet[column] = review_sheet[column].fillna(default)
review_sheet.to_csv(review_path, index=False)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_rows": len(candidate_pool), "loaded_companies": len(all_company_facts),
    "companies_with_annual_anchors": candidate_panel["ticker"].nunique(),
    "annual_periods": len(candidate_panel), "snapshot_times": snapshot_times,
    "tag_candidates_in_priority_order": tag_candidates, "annual_period_days": [350, 380],
    "company_tag_additions": company_tag_additions, "extraction_version": "module7_rev2",
    "min_core_periods_for_review": MIN_CORE_PERIODS, "unit": "USD",
    "filing_rule": "Earliest visible annual revenue filing in the cached data; same-accession metric matching",
    "derivations_applied": False, "accounting_scope_verified": False,
    "final_sample_selected": False, "point_in_time_certified": False,
}
(PROCESSED_DIR / "candidate_module7_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Screening outputs saved in:", PROCESSED_DIR)
